# 04 · Türkçe Puanlayıcı (Anlamsal Benzerlik) İnce Ayarı
`emrecan/bert-base-turkish-cased-mean-nli-stsb-tr` (Apache-2.0) gömme modeli, öğretmen puanlı (cevap, anahtar, 0–1 puan) çiftleriyle CoSENTLoss ile uyarlanır. **T4'e rahat sığar.** Çıktı Mac'te CPU/MPS ile puanlamada kullanılır.

> Eğitim verisi: `sinav-okur disaaktar` ile topladığınız düzeltilmiş cevaplardan ya da elle hazırlanmış CSV'den. Az veriyle bile STSb-TR üstüne ince ayar fayda sağlar.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!pip install -q sentence-transformers datasets

## Eğitim çiftlerini hazırla
CSV kolonları: `cevap, anahtar, puan` (puan 0–1 arası normalize). Örnek satır: `"mitokondri enerji üretir","hücrenin enerji santralidir",1.0`

In [ ]:
import pandas as pd
# Kendi dosyanız: /content/drive/MyDrive/puan_ciftleri.csv
df = pd.read_csv('/content/drive/MyDrive/puan_ciftleri.csv')
print(len(df), 'çift'); df.head()

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
model = SentenceTransformer('emrecan/bert-base-turkish-cased-mean-nli-stsb-tr')
ornekler = [InputExample(texts=[r.cevap, r.anahtar], label=float(r.puan))
            for r in df.itertuples()]
yukleyici = DataLoader(ornekler, shuffle=True, batch_size=16)
kayip = losses.CoSENTLoss(model)

In [ ]:
model.fit(train_objectives=[(yukleyici, kayip)], epochs=4, warmup_steps=50,
          output_path='puanlayici-tr', show_progress_bar=True)

In [ ]:
# STSb-TR üstünde hızlı değerlendirme (opsiyonel)
from datasets import load_dataset
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
try:
    sts = load_dataset('emrecan/stsb-mt-turkish', split='test')
    ev = EmbeddingSimilarityEvaluator([x['sentence1'] for x in sts],
        [x['sentence2'] for x in sts], [x['score']/5 for x in sts])
    print('Spearman:', ev(model))
except Exception as e:
    print('değerlendirme atlandı:', e)

In [ ]:
!zip -qr /content/drive/MyDrive/puanlayici-tr.zip puanlayici-tr
print('Mac: gomlemci_yukle(\'./puanlayici-tr\') ile kullanın')